# Full run on Kaggle (free 2× T4, FP32)

Runs `pipeline.py` (GPU check → trial → recognition + P0 advice + variance → P-Oracle grid) on Kaggle's free **GPU T4 x2**.
Why this route and not Colab or FP16: see `02-llm/scripts/README.md`, "Free GPU: Kaggle 2× T4".

**Study precision: FP32, tensor parallel 2.** Every full-run command uses `--precision fp32 --tp 2 --gpu-mem 0.95 --max-batched-tokens 1024 --max-num-seqs 32 --cuda-graphs decode`, in every session.

## Before the first session

1. **Settings** (right panel): Accelerator **GPU T4 x2**, Internet **on**, Persistence **off**.
2. **Add-ons → Secrets:** add `HF_TOKEN` (the Hugging Face account must have accepted the Gemma license).
3. **Input → Upload → New dataset**, **Private**: upload `01-preprocessing/silver/advice_pairs.parquet`. Name the dataset `payo-silver`. It is study data: never make it public.

## Splitting across two Kaggle accounts

To halve the wall time, run Gemma on one account and SEA-LION on another at the same time: set `MODEL = "gemma"` in one copy and `MODEL = "sealion"` in the other, with the same `COMMIT` and `FLAGS`. Then download both `gold/` outputs and, on the main machine, merge them: copy the first into `02-llm/gold/`, then run `uv run python 02-llm/scripts/merge_gold.py <second gold folder>`.

## Blindness rule

Don't open, preview or print the output JSONL files, here or in Kaggle's output viewer, until `finalize-labels` (RUNBOOK). The run prints only counts.

## Each session

Use **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 hours, and the files in `/kaggle/working` become the version's output.
To resume after a session ends, attach that version's output as an input (**Input → Your Work → this notebook**) and set `PREVIOUS` below. `run.py` skips every finished generation.

## Settings

In [ ]:
REPO = "https://github.com/mbchavez27/payo-bench.git"
COMMIT = "main"    # pin the commit the run uses (git rev-parse HEAD on the main machine) and keep it for every session
import glob
# Kaggle mounts datasets at /kaggle/input/<slug>/ or /kaggle/input/datasets/<user>/<slug>/: find the file
SILVER = glob.glob("/kaggle/input/**/advice_pairs.parquet", recursive=True)[0]
PREVIOUS = None    # resume: e.g. "/kaggle/input/<this-notebook>/gold" (the earlier version's output)
WORK = "/tmp/payo-bench"   # outside /kaggle/working, so .env and the silver data never land in the saved output
FLAGS = "--precision fp32 --tp 2 --gpu-mem 0.95 --max-batched-tokens 1024 --max-num-seqs 32 --cuda-graphs decode"   # same for every session and machine
MODEL = "both"    # split across two Kaggle accounts: "gemma" on one, "sealion" on the other

## GPUs
Should list two Tesla T4 cards, 15 GB each.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv

## Repository, token and silver data

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

if not Path(WORK).exists():
    subprocess.run(["git", "clone", REPO, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "-q", "origin"], check=True)  # a rerun in the same session sees new commits
subprocess.run(["git", "-C", WORK, "checkout", COMMIT], check=True)
print(subprocess.run(["git", "-C", WORK, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout)

Path(WORK, ".env").write_text(f"HF_TOKEN={UserSecretsClient().get_secret('HF_TOKEN')}\n", encoding="utf-8")
silver = Path(WORK, "01-preprocessing/silver/advice_pairs.parquet")
silver.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(SILVER, silver)

## Resume
Copies the previous session's outputs into `02-llm/gold/`, so finished generations are skipped. Nothing is opened.

In [ ]:
if PREVIOUS:
    shutil.copytree(PREVIOUS, Path(WORK, "02-llm/gold"), dirs_exist_ok=True)
    print("resumed from", PREVIOUS)
else:
    print("fresh start")

## Environment
uv installs Python 3.12 and the `full` extra (vLLM). This takes a few minutes.

In [ ]:
!pip install -q uv
%cd {WORK}
!uv sync --extra full

## Run
The GPU check prints both T4s and the precision, then the trial runs (3 posts + 1 grid post into `/tmp/payo-trial`), then the real runs. If the session ends first, rerun the notebook with `PREVIOUS` set.

In [ ]:
!uv run python pipeline.py {FLAGS} --model {MODEL}

## Keep the outputs
Runs even if the pipeline stopped (`!` commands don't halt the notebook). Copies `02-llm/gold/` into `/kaggle/working`, so it is saved with this version. Download it to the main machine **without opening it**.

In [ ]:
out = Path("/kaggle/working/gold")
shutil.copytree(Path(WORK, "02-llm/gold"), out, dirs_exist_ok=True)
print(sum(1 for _ in out.rglob("*.jsonl")), "JSONL files saved under", out)